In [ ]:
import cv2
import numpy as np

def process_both_lungs(image_path: str, black_threshold: int = 15) -> np.ndarray:
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Не удалось загрузить изображение: {image_path}")
    
    h, w = img.shape
    center_x, center_y = w / 2.0, h / 2.0
    mid_col = int(w // 2)

    #черные пиксели
    black_mask = (img <= black_threshold).astype(np.uint8) * 255

    #кластеры
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        black_mask, connectivity=8
    )
    
    left_best_label = None
    left_min_dist = float('inf')
    
    right_best_label = None
    right_min_dist = float('inf')

    #лучший кластер отдельно для левой и правой части
    for label in range(1, num_labels):
        cx, cy = centroids[label]
        dist = np.sqrt((cx - center_x) ** 2 + (cy - center_y) ** 2)
        
        #центроид кластера в левой половине
        if cx < mid_col:
            if dist < left_min_dist:
                left_min_dist = dist
                left_best_label = label
        #центроид кластера в правой половине
        else:
            if dist < right_min_dist:
                right_min_dist = dist
                right_best_label = label

    #оставляем только выбранные 2 кластера
    result_img = img.copy()
    valid_labels = {left_best_label, right_best_label} - {None}

    #красим пиксели в белый
    keep_mask = np.isin(labels, list(valid_labels))
    remove_mask = (black_mask > 0) & (~keep_mask)
    
    result_img[remove_mask] = 255

    return result_img